# Handling Categorical Data with One-Hot Encoding

---
## 1.&nbsp; The Problem with Text Data 📝

In our previous notebooks, we've built models using numerical features like 'Age' and 'Fare'. But we've been ignoring important text-based columns like 'Sex' or 'Embarked'.

Why? Because scikit-learn models, and machine learning models in general, are built on mathematics. They simply don't understand text strings like "male" or "female".

To use this data, we need to convert it into a numerical format. This process is called categorical encoding. We're going to explore the most common method: One-Hot Encoding.

First, we'll walk through the steps manually to see exactly what's happening. Then, we'll show you the correct, automated way to do it using a `ColumnTransformer`, which is a powerful tool for building complex pipelines.

---
## 2.&nbsp; Setup and Data Loading ⚙️

Let's start by loading our libraries and the familiar Titanic dataset. We'll do the same basic preparation as last time.

In [1]:
import pandas as pd
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

# This helps visualise our pipelines
set_config(transform_output='pandas')

# Reading data
url = "https://drive.google.com/file/d/1g3uhw_y3tboRm2eYDPfUzXXsw8IOYDCy/view?usp=sharing"
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
data = pd.read_csv(path)

# X and y creation
X = data.drop(columns=["PassengerId", "Name", "Ticket"])
y = X.pop("Survived")

# Feature Engineering
X.loc[:, "Cabin"] = X.Cabin.str[0]

# Data splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

Now, let's inspect the training data with `.info()`.

In [2]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 329 to 510
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Pclass    712 non-null    int64  
 1   Sex       712 non-null    object 
 2   Age       564 non-null    float64
 3   SibSp     712 non-null    int64  
 4   Parch     712 non-null    int64  
 5   Fare      712 non-null    float64
 6   Cabin     156 non-null    object 
 7   Embarked  710 non-null    object 
dtypes: float64(2), int64(3), object(3)
memory usage: 50.1+ KB


We can see three columns ('Sex', 'Cabin', 'Embarked') are of type 'object'. These are our categorical features. 'Age' (a float) and 'Cabin' also have missing values (`NaN`) that we'll need to handle.

In [18]:
data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [17]:
data["Embarked"].value_counts()


,count
Embarked,
S,644
C,168
Q,77


---
## 3.&nbsp; The "Manual" Approach: Understanding the Steps 👨‍🔧

To understand what a pipeline will do for us later, it's helpful to perform the steps one by one. This isn't how you'd normally build a model, as it's prone to errors and data leakage, but it's great for learning.

### 3.1. Separate Imputation Strategies

We have missing values in both numerical ('Age') and categorical ('Cabin', 'Embarked') columns. We can't use the same strategy for both. You can't find the 'mean' of a cabin letter.

* **For numbers:** We'll fill `NaN` with the column's mean.
* **For categories:** We'll create a new category, **`"N_A"`**, to represent 'Not Available'. This is a simple and effective strategy.

First, let's fill in the missing values in the numberical columns with the mean value.

In [3]:
# 1. Select numerical columns from the training data
X_train_num = X_train.select_dtypes(include="number")

# 2. Create and fit numerical imputer
num_imputer = SimpleImputer(strategy="mean")
X_num_imputed = num_imputer.fit_transform(X_train_num)

X_num_imputed.head()

,Pclass,Age,SibSp,Parch,Fare
329,1.0,16.0,0.0,1.0,57.9792
749,3.0,31.0,0.0,0.0,7.7500
203,3.0,45.5,0.0,0.0,7.2250
421,3.0,21.0,0.0,0.0,7.7333
97,1.0,23.0,0.0,1.0,63.3583


Now we can fill all of the missing values in the categorical columns with `"N_A"`.

In [4]:
# 1. Select categorical columns from the training data
X_train_cat = X_train.select_dtypes(exclude="number")

# 2. Create and fit categorical imputer
cat_imputer = SimpleImputer(strategy="constant", fill_value="N_A")
X_cat_imputed = cat_imputer.fit_transform(X_train_cat)

X_cat_imputed.head()

,Sex,Cabin,Embarked
329,female,B,C
749,male,N_A,Q
203,male,N_A,C
421,male,N_A,Q
97,male,D,C


### 3.2. One-Hot Encoding

Great. All our data is present, but the categorical data is still text. Now we'll use `OneHotEncoder` to convert it.

This transformer finds all unique categories in a column (e.g., 'male', 'female') and creates a new binary column for each one (e.g., 'Sex_male', 'Sex_female').

In [5]:
# 1. Initialise the OneHotEncoder
my_onehot = OneHotEncoder(sparse_output=False,
                          handle_unknown='ignore')

# 2. Fit it on the imputed categorical data
my_onehot.fit(X_cat_imputed)

# 3. Transform the data
X_cat_imputed_onehot = my_onehot.transform(X_cat_imputed)

A quick note on these parameters:

* **`sparse_output=False`**: By default, this transformer creates a 'sparse matrix' (a memory-efficient object for when you have mostly zeros). We're setting it to `False` so we get a regular pandas DataFrame, which is easier to look at.
* **`handle_unknown='ignore'`**: This is important. The encoder learns categories from the training data. If it sees a new category in the test data (one it's never seen before), it would normally throw an error. This tells it to just put a `0` in all columns for that category and move on.

Let's see the result. Notice how 'Sex' (2 categories), 'Cabin' (9 categories), and 'Embarked' (4 categories) have been "expanded" into $2+9+4=15$ new columns.

In [6]:
X_cat_imputed_onehot.head()

,Sex_female,Sex_male,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_N_A,Cabin_T,Embarked_C,Embarked_N_A,Embarked_Q,Embarked_S
329,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
749,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
203,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
421,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
97,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


### 3.3. Recombining Our Data

Finally, we join our new encoded columns with our imputed numerical columns to create our final, model-ready training set.

In [7]:
X_imputed = pd.concat([X_cat_imputed_onehot, X_num_imputed], axis=1)

X_imputed.head()

,Sex_female,Sex_male,Cabin_A,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_N_A,Cabin_T,Embarked_C,Embarked_N_A,Embarked_Q,Embarked_S,Pclass,Age,SibSp,Parch,Fare
329,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,16.0,0.0,1.0,57.9792
749,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0,31.0,0.0,0.0,7.7500
203,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,3.0,45.5,0.0,0.0,7.2250
421,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0,21.0,0.0,0.0,7.7333
97,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,23.0,0.0,1.0,63.3583


That was a lot of work. We had to manually keep track of `X_num_imputed`, `X_cat_imputed_onehot`, and `X_imputed`.

This process is messy and makes it very easy to make mistakes (like accidentally fitting the imputer on the test set). There's a much better way.

---
## 4.&nbsp; The "Automated" Approach - Pipelines with Branches 🚀

We can do all those steps at once using a `ColumnTransformer`. This is the standard scikit-learn tool for this exact job.

It allows us to define separate "branches" or mini-pipelines, one for our numerical columns and one for our categorical columns, and tell it which columns to send down each path. It runs all the steps and automatically joins the results at the end.

### 4.1. Defining the 'Branches'

First, we'll select our column names. Then, we'll create a `numeric_pipe` (which only needs to impute) and a `categoric_pipe` (which needs to impute and one-hot encode).

In [8]:
from sklearn.compose import make_column_transformer

# Select categorical and numerical column names
X_cat_columns = X.select_dtypes(exclude="number").columns
X_num_columns = X.select_dtypes(include="number").columns

# Create numerical pipeline
numeric_pipe = make_pipeline(
    SimpleImputer(strategy="mean")
)

# Create categorical pipeline
categoric_pipe = make_pipeline(
    SimpleImputer(strategy="constant",
                  fill_value="N_A"),
    OneHotEncoder(sparse_output=False,
                  handle_unknown='ignore')
)

### 4.2. Creating the Preprocessor with `make_column_transformer`

Now we plug our two mini-pipelines into `make_column_transformer`, telling it which pipe to use for which list of columns.

In [9]:
preprocessor = make_column_transformer(
    (numeric_pipe, X_num_columns),
    (categoric_pipe, X_cat_columns)
)

preprocessor

ColumnTransformer(transformers=[('pipeline-1',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer())]),
                                 Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                ('pipeline-2',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='N_A',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])

### 4.3. Building the Full Pipeline

This `preprocessor` is just another scikit-learn transformer. We can put it as the first step in a new, full pipeline, with our `DecisionTreeClassifier` as the final step.

In [10]:
full_pipeline = make_pipeline(preprocessor,
                              DecisionTreeClassifier(random_state=42))

full_pipeline

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer())]),
                                                  Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='N_A',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(random_state=42))])

Look at that! It's a clean, single object that contains all our logic.

Now we can just fit it once on the raw `X_train` data. It will automatically handle splitting the columns, imputing, encoding, and training the model, all in one go.

In [11]:
full_pipeline.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('pipeline-1',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer())]),
                                                  Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                                 ('pipeline-2',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(fill_value='N_A',
                                                                                 strategy='constant')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])),
                ('decisiontreeclassifier',
                 DecisionTreeClassifier(random_state=42))])

---
## 5.&nbsp; Using Our New Pipeline with `GridSearchCV` 🧠

This is where the pipeline really shows its power. We can use `GridSearchCV` to tune any part of the pipeline, whether it's the model or a step in the preprocessor.

We just use the `__` (double-underscore) syntax to name the parameter. For example:
* `decisiontreeclassifier__max_depth`: Targets the `max_depth` of our model.
* `columntransformer__pipeline-1__simpleimputer__strategy`: This targets the `strategy` parameter of the `simpleimputer` inside `pipeline-1` (our numeric pipe) which is inside the `columntransformer`.

Let's try tuning the tree's depth and the numerical imputer's strategy at the same time.

In [12]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid
param_grid = {
    "columntransformer__pipeline-1__simpleimputer__strategy":["mean", "median"],
    "decisiontreeclassifier__max_depth": range(2, 14, 2),
    "decisiontreeclassifier__min_samples_leaf": range(3, 12, 2)
}

# Define GridSearchCV
search = GridSearchCV(full_pipeline, # using the pipeline from the previous cell
                      param_grid,
                      cv=5,
                      verbose=1,
                      n_jobs=-1)

search.fit(X_train, y_train)

print(f"Best score: {search.best_score_:.4f}")
print("Best parameters found:")
print(search.best_params_)

Fitting 5 folds for each of 60 candidates, totalling 300 fits
Best score: 0.8062
Best parameters found:
{'columntransformer__pipeline-1__simpleimputer__strategy': 'mean', 'decisiontreeclassifier__max_depth': 4, 'decisiontreeclassifier__min_samples_leaf': 3}


It looks like a `max_depth` of 4, `min_samples_leaf` of 3, and using the 'mean' strategy gave us the best cross-validation score.

And we did it all without ever manually imputing or encoding anything!

---
## 6.&nbsp; (Bonus) Inspecting Pipeline Steps 🕵️‍♀️

Sometimes you want to "look inside" the pipeline. You can access the fitted steps using their names. Since we used `make_pipeline`, the steps are named after their class (e.g., `columntransformer`).

In [13]:
# Access the 'columntransformer' step
full_pipeline.named_steps.columntransformer

ColumnTransformer(transformers=[('pipeline-1',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer())]),
                                 Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], dtype='object')),
                                ('pipeline-2',
                                 Pipeline(steps=[('simpleimputer',
                                                  SimpleImputer(fill_value='N_A',
                                                                strategy='constant')),
                                                 ('onehotencoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 Index(['Sex', 'Cabin', 'Embarked'], dtype='object'))])

You can go even deeper. Let's get the names of the final one-hot encoded features from the `categoric_pipe` (which was automatically named 'pipeline-2' inside the `columntransformer`).

In [14]:
(
  full_pipeline
  .named_steps.columntransformer
  .named_transformers_['pipeline-2']
  .named_steps.onehotencoder
  .get_feature_names_out()
 )

array(['Sex_female', 'Sex_male', 'Cabin_A', 'Cabin_B', 'Cabin_C',
       'Cabin_D', 'Cabin_E', 'Cabin_F', 'Cabin_G', 'Cabin_N_A', 'Cabin_T',
       'Embarked_C', 'Embarked_N_A', 'Embarked_Q', 'Embarked_S'],
      dtype=object)

This is very useful for understanding what your model is actually "seeing".

(If you want even more readable names, you can use the `Pipeline` and `ColumnTransformer` classes directly instead of `make_pipeline` and `make_column_transformer`. This lets you provide custom names for each step, like 'preprocessor' or 'cat-pipe').

---
## 7.&nbsp; Key Takeaways 🔑

This was a big step. Here's what we've learned:

* ML models need **numerical data**. Categorical (text) data must be converted.
* **One-Hot Encoding** is a standard technique that creates a new binary (0 or 1) column for each unique category.
* Handling numerical and categorical data requires different preprocessing steps (e.g., `SimpleImputer(strategy='mean')` vs. `SimpleImputer(strategy='constant')`).
* **`ColumnTransformer`** is the best tool for the job. It lets you define 'branches' (like `numeric_pipe` and `categoric_pipe`) and applies them to different columns automatically.
* You can put a `ColumnTransformer` inside a `Pipeline` with your model to create a single, clean object.
* You can tune hyperparameters in any part of a pipeline using `GridSearchCV` and the `__` (double-underscore) syntax.

---
## 8.&nbsp; Challenge 🙂

In a new notebook, apply everything you have learned here (using a `ColumnTransformer` and a full `Pipeline`) to the Housing project.

That dataset has many categorical features. You'll need to:
1.  Identify all the numerical and categorical columns.
2.  Create a `numeric_pipe`.
3.  Create a `categoric_pipe` with an imputer and `OneHotEncoder`.
4.  Combine them in a `ColumnTransformer`.
5.  Build a `full_pipeline` with your preprocessor and a model.
6.  Use `GridSearchCV` to find the best parameters.

In [15]:
# Good luck!